# Modelo por regresión lineal múltiple

In [1]:
## CARGA DE LIBRERIAS ##
import pandas as pd
import numpy as np
import matplotlib as plt #?
import seaborn as sns #?
import pickle

In [2]:
## CARGA DE DATASET ##
df = pd.read_csv('~/Documents/Universidad/TFM/TFM_imarcospu/Dataset/SDN-DDoS_Traffic_Dataset.csv') # Ver si se puede hacer una carga mediante pickle para no tener que cargar el fichero de forma local
# Observamos los primeros datos
df.head(5)

,switch,host,src_ip,dst_ip,pkt_count,byte_count,duration,duration_nsec,tot_duration,flows,...,port_no,tx_bytes,rx_bytes,tx_kbps,rx_kbps,tot_kbps,delay,jitter,packet_loss_rate,label
0,10,23,10.0.0.17,10.0.0.14,76411,689600385,331,261597151,1.058522e+10,7,...,3,44721489,59528897,19662,6719,10363,96.230855,5.658548,2.994103,1
1,11,23,10.0.0.22,10.0.0.21,89043,994428451,1455,717489475,1.787452e+10,11,...,3,17090261,49017915,3780,16735,2077,98.699707,11.251787,3.106546,0
2,5,11,10.0.0.12,10.0.0.8,184943,432710323,71,719260236,1.196684e+10,11,...,3,44053439,110946853,10365,17183,13339,89.542596,11.306913,4.055693,0
3,6,4,10.0.0.16,10.0.0.10,137786,623653042,2520,825085778,5.273923e+09,4,...,3,9709880,633368255,1003,6876,14905,61.076879,0.890531,0.317505,1
4,3,18,10.0.0.8,10.0.0.14,231543,1023396293,488,805586380,6.316448e+10,5,...,1,16546692,407867647,10324,14634,13281,45.331949,19.911099,0.050583,1


La variable objetivo que se encargará de realizar las predicciones de tráfico será tot_kbps, que indica el total de ancho de banda utilizado en dicha traza de tráfico. De esta forma, dada una traza, el controlador SDN podrá asignar un ancho de banda específico para dicha transmisión, gestionando y adaptando los recursos de red de una forma mucho más eficiente. El dataset cuenta con una columna label que actúa como un flag que toma valores 0 o 1, realizando una diferenciación en dos clases de tráfico: normal y de ataque. Dado que nuestro objetivo con este modelo es realizar predicciones de tráfico, escogemos únicamente las trazas que sean de tráfico normal, excluyendo el tráfico de ataque.

In [3]:
df = df[df['label'] == 0]
# eliminamos columna label ya que no nos aporta información (siempre es igual a 0)
del df['label']
df.head(5)

# eliminamos también columna switch y host ya que tampoco aportan información relevante ya que el origen y destino viene dado
# por src_ip y dst_ip
del df['switch']
del df['host']
df.head(5)

,src_ip,dst_ip,pkt_count,byte_count,duration,duration_nsec,tot_duration,flows,packet_per_massg,pktper_flow,...,Protocol,port_no,tx_bytes,rx_bytes,tx_kbps,rx_kbps,tot_kbps,delay,jitter,packet_loss_rate
1,10.0.0.22,10.0.0.21,89043,994428451,1455,717489475,1.787452e+10,11,12351,22444.81276,...,TCP,3,17090261,49017915,3780,16735,2077,98.699707,11.251787,3.106546
2,10.0.0.12,10.0.0.8,184943,432710323,71,719260236,1.196684e+10,11,16104,16064.64412,...,UDP,3,44053439,110946853,10365,17183,13339,89.542596,11.306913,4.055693
6,10.0.0.7,10.0.0.21,63104,1220845710,175,495805933,9.217866e+10,1,8161,28642.14835,...,UDP,3,56611685,344675008,19262,154,8575,19.867770,2.365249,1.374554
8,10.0.0.8,10.0.0.14,242569,107067762,1537,542212385,9.101546e+10,10,10775,23190.30861,...,TCP,3,51292099,262907917,3439,6403,8399,65.094153,0.064573,1.779630
9,10.0.0.23,10.0.0.1,222976,856171915,1460,24596441,3.854735e+10,9,6952,17098.69557,...,ICMP,2,4634550,84477460,10998,3899,4823,84.415030,17.579673,4.185496


## Datos de entrenamiento/validación/test

Para elaborar y validar el modelo, dividimos la totalidad de los datos en tres partes: 60% para entrenar el modelo, un 20% para validarlo y otro 20% para testear el modelo. Después debemos excluir la variable objetivo del dataframe, así evitaremos utilizarla para los cálculos del modelo.

In [4]:
n = len(df) # numero de registros

n_val = int(n*0.2) # numero de registros para validacion
n_test = int(n*0.2) # numero de registros para test
n_train = n - n_val - n_test # numero de registros para entrenamiento

# escogemos los registros acordados del dataframe
df_train = df.iloc[:n_train]
df_val = df.iloc[n_train:n_train+n_val]
df_test = df.iloc[n_train+n_val:]

# barajamos los datos para evitar linealidades estadísticas (datos no consecutivos)

index = np.arange(n) # vector de 0 hasta n
np.random.seed(5) # establecemos semilla
np.random.shuffle(index) # barajamos el vector

# barajamos el dataframe
df_train = df.iloc[index[:n_train]]
df_val = df.iloc[index[n_train:n_train+n_val]]
df_test = df.iloc[index[n_train+n_val:]]

# reseteamos indices para que vayan de 0 a n_*
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_val.reset_index(drop=True)

# seleccionamos nuestra variable objetivo del dataframe
y_train = df_train.tot_kbps.values
y_val = df_val.tot_kbps.values
y_test = df_test.tot_kbps.values

# eliminamos del dataframe la variable objetivo, evitando utilizarla para los cálculos del modelo
del df_train['tot_kbps']
del df_val['tot_kbps']
del df_test['tot_kbps']

## Análisis exploratorio de datos

Debemos realizar un análisis al dataframe obtenido, comprobando que no tenga datos erróneos que nos entorpezcan a la hora de realizar el modelo. Comprobaremos el nombre de las columnas, el tipo y la codificación de las mismas, así como el escalado (normalización) de los datos. Encapsulamos todo en una función para aplicar a los tres tipos de dataframes

In [5]:
from sklearn.preprocessing import StandardScaler

def prepara_X(df):
    df = df.copy()
    
    # ponemos nombres de columna en minuscula
    df.columns = df.columns.str.lower()
    df.columns
    
    # observamos las cabeceras y sus tipos, viendo que hay tres campos que son de tipo object (src_ip, dst_ip, protocol)
    # se tratan de variables categóricas (string) que debemos transformar para que nuestro modelo ML lo pueda interpretar
    df.dtypes
    
    # vemos que solo hay 24 direcciones IP en total, mientras que protocolos solo hay tres.
    df[['src_ip','dst_ip','protocol']].nunique()
    
    # realizamos codificaciones para la columna protocol mediante one-hot encoding. 
    df['protocol_tcp'] = (df.protocol == 'TCP').astype('int')
    df['protocol_udp'] = (df.protocol == 'UDP').astype('int')
    df['protocol_icmp'] = (df.protocol == 'ICMP').astype('int')
        
    del df['protocol']
    
    df['src_ip_1'] = (df.src_ip == '10.0.0.1').astype('int')
    df['src_ip_2'] = (df.src_ip == '10.0.0.2').astype('int')
    df['src_ip_3'] = (df.src_ip == '10.0.0.3').astype('int')
    df['src_ip_4'] = (df.src_ip == '10.0.0.4').astype('int')
    df['src_ip_5'] = (df.src_ip == '10.0.0.5').astype('int')
    df['src_ip_6'] = (df.src_ip == '10.0.0.6').astype('int')
    df['src_ip_7'] = (df.src_ip == '10.0.0.7').astype('int')
    df['src_ip_8'] = (df.src_ip == '10.0.0.8').astype('int')
    df['src_ip_9'] = (df.src_ip == '10.0.0.9').astype('int')
    df['src_ip_10'] = (df.src_ip == '10.0.0.10').astype('int')
    df['src_ip_11'] = (df.src_ip == '10.0.0.11').astype('int')
    df['src_ip_12'] = (df.src_ip == '10.0.0.12').astype('int')
    df['src_ip_13'] = (df.src_ip == '10.0.0.13').astype('int')
    df['src_ip_14'] = (df.src_ip == '10.0.0.14').astype('int')
    df['src_ip_15'] = (df.src_ip == '10.0.0.15').astype('int')
    df['src_ip_16'] = (df.src_ip == '10.0.0.16').astype('int')
    df['src_ip_17'] = (df.src_ip == '10.0.0.17').astype('int')
    df['src_ip_18'] = (df.src_ip == '10.0.0.18').astype('int')
    df['src_ip_19'] = (df.src_ip == '10.0.0.19').astype('int')
    df['src_ip_20'] = (df.src_ip == '10.0.0.20').astype('int')
    df['src_ip_21'] = (df.src_ip == '10.0.0.21').astype('int')
    df['src_ip_22'] = (df.src_ip == '10.0.0.22').astype('int')
    df['src_ip_23'] = (df.src_ip == '10.0.0.23').astype('int')
    df['src_ip_24'] = (df.src_ip == '10.0.0.24').astype('int')
    
    del df['src_ip']

    df['dst_ip_1'] = (df.dst_ip == '10.0.0.1').astype('int')
    df['dst_ip_2'] = (df.dst_ip == '10.0.0.2').astype('int')
    df['dst_ip_3'] = (df.dst_ip == '10.0.0.3').astype('int')
    df['dst_ip_4'] = (df.dst_ip == '10.0.0.4').astype('int')
    df['dst_ip_5'] = (df.dst_ip == '10.0.0.5').astype('int')
    df['dst_ip_6'] = (df.dst_ip == '10.0.0.6').astype('int')
    df['dst_ip_7'] = (df.dst_ip == '10.0.0.7').astype('int')
    df['dst_ip_8'] = (df.dst_ip == '10.0.0.8').astype('int')
    df['dst_ip_9'] = (df.dst_ip == '10.0.0.9').astype('int')
    df['dst_ip_10'] = (df.dst_ip == '10.0.0.10').astype('int')
    df['dst_ip_11'] = (df.dst_ip == '10.0.0.11').astype('int')
    df['dst_ip_12'] = (df.dst_ip == '10.0.0.12').astype('int')
    df['dst_ip_13'] = (df.dst_ip == '10.0.0.13').astype('int')
    df['dst_ip_14'] = (df.dst_ip == '10.0.0.14').astype('int')
    df['dst_ip_15'] = (df.dst_ip == '10.0.0.15').astype('int')
    df['dst_ip_16'] = (df.dst_ip == '10.0.0.16').astype('int')
    df['dst_ip_17'] = (df.dst_ip == '10.0.0.17').astype('int')
    df['dst_ip_18'] = (df.dst_ip == '10.0.0.18').astype('int')
    df['dst_ip_19'] = (df.dst_ip == '10.0.0.19').astype('int')
    df['dst_ip_20'] = (df.dst_ip == '10.0.0.20').astype('int')
    df['dst_ip_21'] = (df.dst_ip == '10.0.0.21').astype('int')
    df['dst_ip_22'] = (df.dst_ip == '10.0.0.22').astype('int')
    df['dst_ip_23'] = (df.dst_ip == '10.0.0.23').astype('int')
    df['dst_ip_24'] = (df.dst_ip == '10.0.0.24').astype('int')
    
    del df['dst_ip']
    
    # comprobamos que no existan valores nulos
    df.isnull().sum()

    # estandarizamos todos nuestros datos menos las variables binarias con StandardScaler()
    features_num = ['pkt_count', 'byte_count', 'duration', 'duration_nsec', 'tot_duration',
           'flows', 'packet_per_massg', 'pktper_flow', 'byte_per_flow', 'pkt_rate',
           'pair_flow', 'port_no', 'tx_bytes', 'rx_bytes', 'tx_kbps', 'rx_kbps',
           'delay', 'jitter', 'packet_loss_rate']
    features_cat = ['protocol_tcp',
           'protocol_udp', 'protocol_icmp', 'src_ip_1', 'src_ip_2', 'src_ip_3',
           'src_ip_4', 'src_ip_5', 'src_ip_6', 'src_ip_7', 'src_ip_8', 'src_ip_9',
           'src_ip_10', 'src_ip_11', 'src_ip_12', 'src_ip_13', 'src_ip_14',
           'src_ip_15', 'src_ip_16', 'src_ip_17', 'src_ip_18', 'src_ip_19',
           'src_ip_20', 'src_ip_21', 'src_ip_22', 'src_ip_23', 'src_ip_24',
           'dst_ip_1', 'dst_ip_2', 'dst_ip_3', 'dst_ip_4', 'dst_ip_5', 'dst_ip_6',
           'dst_ip_7', 'dst_ip_8', 'dst_ip_9', 'dst_ip_10', 'dst_ip_11',
           'dst_ip_12', 'dst_ip_13', 'dst_ip_14', 'dst_ip_15', 'dst_ip_16',
           'dst_ip_17', 'dst_ip_18', 'dst_ip_19', 'dst_ip_20', 'dst_ip_21',
           'dst_ip_22', 'dst_ip_23', 'dst_ip_24']
    
    df_num = df[features_num]
    df_cat = df[features_cat]
    
    normalizador = StandardScaler()
    df_norm = normalizador.fit_transform(df_num[features_num])

    # concatenamos
    columnas = features_num + features_cat
    X = pd.DataFrame(np.concatenate([df_norm, df_cat], axis=1), columns=columnas)
    
    return X

## Entrenamiento del modelo de regresion lineal

Un modelo lineal debe seguirse por la fórmula de regresión lineal (ver memoria ...)

In [6]:
# implementamos el modelo ...
def modelo_reg_lineal(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0]) # regularizacion
    
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

# error cuadratico medio
def rmse(y, y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()

    return np.sqrt(mse)

In [10]:
# pasamos a obtener las predicciones del modelo
#X_train = prepara_X(df_train) # entrenamos
#w0, w = modelo_reg_lineal(X_train, y_train, r=0.0001)

#X_val = prepara_X(df_val) # validamos
#y_pred = w0 + X_val.dot(w) # validamos
#rmse(y_val, y_pred)

from sklearn.linear_model import LinearRegression

model = LinearRegression()

X_train = prepara_X(df_train)
X_val = prepara_X(df_val)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
rmse(y_val,y_pred)

y_val[10], y_pred[10]

(np.int64(18798), np.float64(11546.894252050663))